# Copper price forecasting: LSTM and simulated annealing

Reproduction with the attached 2006–2026 CSV. Run All after choosing `MODE` in the next code cell. Quick results are diagnostic; use `FULL_RESEARCH_MODE` for the 100 epoch comparison.

## 1. Research objective

Compare naive persistence, fixed vanilla LSTM, the paper's published SA-LSTM configuration, and newly optimized SA-LSTM on three chronological splits. Forecast Copper at the next joint trading session from 22 prior sessions of WTI, Gold and Silver. Use validation only for model selection.

## 2. Paper methodology audit

| Component | Paper evidence | This notebook |
|---|---|---|
| Data | Investing.com, 1990–2009, 4,870 rows (p. 8) | Attached CSV, actual date span checked |
| Features | WTI, Gold, Silver chosen by Spearman > 0.8 (p. 8) | Same three inputs; train-only Spearman audit |
| Inputs | Three named indicators, but Figure 12 has 6,864 parameters in first 39-unit LSTM, implying four features | Three named indicators; Figure 12 discrepancy documented |
| Lookback | Figure 12 output shape `(None, 22, 39)` (p. 15) | 22 timesteps |
| Target horizon | Not specified precisely | Next joint trading session |
| Scaling | MinMax [0,1] (p. 8); fit scope unclear | Fit on train only |
| Architecture | Two LSTMs, Dense(32), Dropout, Dense(1) (Fig. 12); 39/111, dropout .2 (Table 4) | Same shape; Dense ReLU, output linear and Adam are explicit assumptions |
| SA | Eq. 7–11; cooling .97 in method, .95 in experiment; 650 scenarios claimed | Latent-coordinate Eq. 7–8, corrected Metropolis sign, .95 cooling, configured smaller budget |
| Training | Batch 64, 100 epochs; validation handling unclear | 100 epochs in full mode; chronology held fixed |

The paper's printed `exp(+ΔE/T)` for a worse candidate exceeds one; code uses `exp(-ΔE/T)`. See `paper_reproduction_audit.md` for the full evidence and prototype audit.

## 3. Environment and configuration

Attach the Kaggle dataset, set `MODE`, then Run All. The notebook is self contained and makes no network requests.

In [9]:
"""Standalone code embedded verbatim in copper_reproduction_kaggle.ipynb.

Only a local CSV is read. Search receives validation data, never test labels.
"""
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

DATA_PATH = Path("/kaggle/input/datasets/lancelot71000/copper-8-indicators-2006-2026/copper_8_indicators_2006_2026.csv")
MODE = "FULL_RESEARCH_MODE"  # change to FULL_RESEARCH_MODE for publishable experiments
OUTPUT_ROOT = Path("/kaggle/working")
SEED = 42
FEATURES = ["WTI", "Gold", "Silver"]
CANDIDATES = ["Dow_Jones", "Nasdaq", "Dollar_Index", "TBill_10Y", "Gold", "Silver", "WTI", "Copper_Volume"]
RATIOS = [0.90, 0.925, 0.95]
PAPER_CONFIG = {"lstm_units_1": 39, "lstm_units_2": 111, "learning_rate": 5e-5,
                "batch_size": 64, "epochs": 100, "dropout": 0.2}
VANILLA_CONFIG = {"lstm_units_1": 32, "lstm_units_2": 32, "learning_rate": 1e-3}
LOOKBACK = 22  # Figure 12 gives output shape (None, 22, 39); horizon remains undocumented.
DENSE_ACTIVATION = "linear"  # output activation is undocumented; linear permits values outside training range.
RUN_ABLATION = False
MODES = {
    "QUICK_MODE": {"max_rows": 800, "epochs": 2, "sa_epochs": 2, "sa_evaluations": 4,
                   "proposals_per_temperature": 2},
    "FULL_RESEARCH_MODE": {"max_rows": None, "epochs": 100, "sa_epochs": 20,
                           "sa_evaluations": 100, "proposals_per_temperature": 2},
}
SA = {"initial_temperature": 1.0, "final_temperature": 0.1, "cooling": 0.95,
      "units_min": 8, "units_max": 128, "lr_min": 1e-6, "lr_max": 0.005,
      "proposal_width": 2.0, "initial": {"lstm_units_1": 15, "lstm_units_2": 30,
                                          "learning_rate": 0.0005}}


def run_config(mode=MODE):
    if mode not in MODES:
        raise ValueError(f"Unknown mode: {mode}")
    return dict(MODES[mode])


def fingerprint(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")
    os.replace(tmp, path)




## 4. Load dataset and data quality

In [10]:
def load_data(path, out, max_rows=None):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Attach the Kaggle dataset or set DATA_PATH to the local CSV: {path}")
    source = pd.read_csv(path)
    required = ["Date", "Copper"] + CANDIDATES
    missing_columns = sorted(set(required) - set(source.columns))
    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}")
    original_types = source.dtypes.astype(str).to_dict()
    dates = pd.to_datetime(source["Date"], errors="coerce")
    duplicate_dates = int(dates.duplicated().sum())
    invalid_dates = int(dates.isna().sum())
    numeric = source[["Copper"] + CANDIDATES].apply(pd.to_numeric, errors="coerce")
    numeric = numeric.replace([np.inf, -np.inf], np.nan)
    quality = pd.DataFrame([{
        "column": name, "source_dtype": original_types[name], "missing_or_nonfinite": int(numeric[name].isna().sum()),
        "duplicate_dates": duplicate_dates, "invalid_dates": invalid_dates,
        "rows_source": len(source), "first_date": str(dates.min().date()),
        "last_date": str(dates.max().date())
    } for name in ["Copper"] + CANDIDATES])
    quality = pd.concat([pd.DataFrame([{"column": "Date", "source_dtype": original_types["Date"],
                                        "missing_or_nonfinite": invalid_dates,
                                        "duplicate_dates": duplicate_dates, "invalid_dates": invalid_dates,
                                        "rows_source": len(source), "first_date": str(dates.min().date()),
                                        "last_date": str(dates.max().date())}]), quality], ignore_index=True)
    quality.to_csv(out / "data_quality_report.csv", index=False)
    if invalid_dates or duplicate_dates:
        raise ValueError("Invalid or duplicate Date values; inspect data_quality_report.csv")
    data = numeric.copy()
    data.insert(0, "Date", dates)
    data = data.sort_values("Date").dropna().set_index("Date")
    if max_rows is not None:
        data = data.tail(max_rows).copy()
    if len(data) < 100 or not data.index.is_unique or not data.index.is_monotonic_increasing:
        raise ValueError("Insufficient or unordered complete rows")
    if not np.isfinite(data.to_numpy()).all():
        raise ValueError("Nonfinite data after cleaning")
    quality["rows_complete_used"] = len(data)
    quality["rows_dropped_missing"] = int(numeric.isna().any(axis=1).sum())
    quality.to_csv(out / "data_quality_report.csv", index=False)
    return data, quality




## 5. EDA and train-only Spearman

The correlation table and heatmap are produced by `run_experiment` below.

## 6. Chronological split and preprocessing

Each split fits independent MinMax scalers on its fit segment. A target at index `t` receives inputs only from `[t-22, ..., t-1]`. Validation and test windows may use earlier observations across the boundary, never later observations.

In [11]:
def boundaries(n, ratio, validation_fraction=0.10):
    test_start = int(n * ratio)
    val_start = int(test_start * (1 - validation_fraction))
    if not 0 < val_start < test_start < n:
        raise ValueError("Invalid split")
    return val_start, test_start


def train_spearman(data, fit_end):
    fit = data.iloc[:fit_end]
    result = fit[["Copper"] + CANDIDATES].corr(method="spearman")
    return result


def windows(x, y, dates, start, end, lookback):
    if start < lookback or end <= start:
        raise ValueError("Window outside chronological segment")
    X = np.stack([x[t - lookback:t] for t in range(start, end)]).astype("float32")
    Y = y[start:end].astype("float32")
    assert X.shape == (end - start, lookback, len(FEATURES))
    return X, Y, dates[start:end]


def prepare(data, ratio, lookback=LOOKBACK, include_test=False):
    fit_end, test_start = boundaries(len(data), ratio)
    if fit_end <= lookback:
        raise ValueError("Insufficient training rows")
    fit = data.iloc[:fit_end]
    x_scaler = MinMaxScaler().fit(fit[FEATURES])
    y_scaler = MinMaxScaler().fit(fit[["Copper"]])
    visible = data if include_test else data.iloc[:test_start]
    x = x_scaler.transform(visible[FEATURES]).astype("float32")
    y = y_scaler.transform(visible[["Copper"]]).ravel().astype("float32")
    result = {"train": windows(x, y, visible.index, lookback, fit_end, lookback),
              "val": windows(x, y, visible.index, fit_end, test_start, lookback),
              "x_scaler": x_scaler, "y_scaler": y_scaler, "fit_end": fit_end,
              "test_start": test_start, "ratio": ratio, "lookback": lookback,
              "spearman": train_spearman(data, fit_end),
              "feature_rule": "Fixed WTI, Gold, Silver from paper; Spearman audit on train only"}
    if include_test:
        result["test"] = windows(x, y, visible.index, test_start, len(data), lookback)
        result["previous_copper"] = data["Copper"].shift(1).iloc[test_start:].to_numpy()
    assert result["train"][2].max() < result["val"][2].min()
    if include_test:
        assert result["val"][2].max() < result["test"][2].min()
    assert np.isfinite(result["train"][0]).all() and np.isfinite(result["val"][1]).all()
    return result


def split_summary(data):
    rows = []
    for ratio in RATIOS:
        a, b = boundaries(len(data), ratio)
        for name, start, end in [("train", LOOKBACK, a), ("validation", a, b), ("test", b, len(data))]:
            rows.append({"ratio": ratio, "segment": name, "samples": end - start,
                         "first_target": str(data.index[start].date()), "last_target": str(data.index[end - 1].date())})
    return pd.DataFrame(rows)


def plot_spearman(corr, out):
    corr.to_csv(out / "spearman_train.csv")
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(corr)), corr.columns, rotation=65, ha="right")
    ax.set_yticks(range(len(corr)), corr.index)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(out / "spearman_heatmap.png", dpi=150)
    plt.close(fig)




## 7. LSTM architecture

Two stacked tanh/sigmoid LSTMs, Dense(32, ReLU), Dropout(.2), Dense(1, linear). The last two activations and Adam are implementation assumptions. The output activation can be changed to `tanh` or `sigmoid` for a separate ablation.

In [12]:
def configure_tensorflow():
    import tensorflow as tf
    for gpu in tf.config.list_physical_devices("GPU"):
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
    tf.keras.utils.set_random_seed(SEED)
    return tf


def build_model(n_features, lookback, hp, activation=DENSE_ACTIVATION):
    import tensorflow as tf
    if activation not in {"linear", "tanh", "sigmoid"}:
        raise ValueError("Unsupported output activation")
    model = tf.keras.Sequential([
        tf.keras.layers.Input((lookback, n_features)),
        tf.keras.layers.LSTM(int(hp["lstm_units_1"]), activation="tanh", recurrent_activation="sigmoid", return_sequences=True),
        tf.keras.layers.LSTM(int(hp["lstm_units_2"]), activation="tanh", recurrent_activation="sigmoid"),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(PAPER_CONFIG["dropout"]),
        tf.keras.layers.Dense(1, activation=activation),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=float(hp["learning_rate"])), loss="mse")
    return model


def train_model(ctx, hp, epochs, seed=SEED, activation=DENSE_ACTIVATION, predict_test=False):
    import tensorflow as tf
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(seed)
    Xtr, ytr, _ = ctx["train"]
    Xval, yval, _ = ctx["val"]
    model = build_model(Xtr.shape[-1], ctx["lookback"], hp, activation)
    start = time.perf_counter()
    try:
        history = model.fit(Xtr, ytr, validation_data=(Xval, yval), epochs=epochs,
                            batch_size=PAPER_CONFIG["batch_size"], shuffle=False, verbose=0)
        val_curve = np.asarray(history.history["val_loss"], dtype=float)
        if not np.isfinite(val_curve).all():
            raise FloatingPointError("Training diverged")
        val_pred = model.predict(Xval, verbose=0).ravel()
        if not np.isfinite(val_pred).all() or val_pred.shape != yval.shape:
            raise FloatingPointError("Invalid validation predictions")
        result = {"val_mse_scaled": float(mean_squared_error(yval, val_pred)),
                  "train_seconds": time.perf_counter() - start,
                  "history": history.history}
        if predict_test:
            Xte, yte, dates = ctx["test"]
            pred = model.predict(Xte, verbose=0).ravel()
            if pred.shape != yte.shape or not np.isfinite(pred).all():
                raise FloatingPointError("Invalid test predictions")
            result["prediction_scaled"] = pred
            result["test_dates"] = dates
        return result
    finally:
        del model
        tf.keras.backend.clear_session()
        gc.collect()




## 8. Simulated Annealing

Proposal uses Eq. 7–8 in normalized latent coordinates; neuron counts are rounded and learning rate is searched in log space. Proposal width and bounds are assumptions. Search minimizes normalized validation MSE. A checkpoint contains RNG, current and best states.

In [13]:
def _sigmoid(z):
    return 1.0 / (1.0 + math.exp(-float(np.clip(z, -30, 30))))


def _logit(p):
    p = float(np.clip(p, 1e-6, 1 - 1e-6))
    return math.log(p / (1 - p))


def proposal(current, temperature, rng):
    """Paper Eq 7–8 in normalized latent coordinates; width is an explicit assumption."""
    candidate = {}
    for key in ("lstm_units_1", "lstm_units_2", "learning_rate"):
        if key == "learning_rate":
            lo, hi = math.log(SA["lr_min"]), math.log(SA["lr_max"])
            value = math.log(current[key])
        else:
            lo, hi = SA["units_min"], SA["units_max"]
            value = current[key]
        latent = _logit((value - lo) / (hi - lo))
        temporary = latent + temperature * SA["proposal_width"] * rng.uniform(-1, 1)
        bounded = lo + (hi - lo) * _sigmoid(temporary)
        candidate[key] = float(math.exp(bounded)) if key == "learning_rate" else int(np.clip(round(bounded), lo, hi))
    return candidate


def metropolis(delta, temperature, rng):
    # Paper Eq 10 prints exp(+delta/T), impossible for worse solutions; corrected sign.
    probability = 1.0 if delta <= 0 else math.exp(-delta / temperature)
    return bool(rng.random() < probability), probability


def sa_search(objective, out, identity, mode, budget, seed=SEED):
    """Checkpoint after each evaluated candidate; objective returns (validation MSE, seconds)."""
    checkpoint = out / "sa_checkpoint.json"
    history_path = out / "sa_search_history.csv"
    signature = fingerprint({"identity": identity, "mode": mode, "SA": SA,
                             "search_epochs": budget["sa_epochs"],
                             "proposals_per_temperature": budget["proposals_per_temperature"],
                             "paper_config": PAPER_CONFIG, "seed": seed})
    rng = np.random.default_rng(seed)
    if checkpoint.exists():
        state = json.loads(checkpoint.read_text(encoding="utf-8"))
        if state["signature"] != signature:
            raise RuntimeError("SA checkpoint belongs to another dataset, split or configuration")
        rng.bit_generator.state = state["rng_state"]
    else:
        hp = dict(SA["initial"])
        loss, seconds = objective(hp)
        if not np.isfinite(loss):
            raise FloatingPointError("Initial SA candidate diverged")
        state = {"signature": signature, "current": hp, "current_loss": loss,
                 "best": hp, "best_loss": loss, "evaluations": 1,
                 "total_epochs": budget["sa_epochs"], "total_seconds": seconds,
                 "history": [{"evaluation": 1, "temperature": SA["initial_temperature"],
                              "validation_mse_scaled": loss, "best_mse_scaled": loss,
                              "accepted": True, "acceptance_probability": 1.0,
                              "lstm_units_1": hp["lstm_units_1"], "lstm_units_2": hp["lstm_units_2"],
                              "learning_rate": hp["learning_rate"], "train_seconds": seconds}]}
        state["rng_state"] = rng.bit_generator.state
        atomic_json(checkpoint, state)
    max_evals = budget["sa_evaluations"]
    while state["evaluations"] < max_evals:
        proposal_number = state["evaluations"] - 1
        temperature = SA["initial_temperature"] * SA["cooling"] ** (
            proposal_number // budget["proposals_per_temperature"])
        if temperature < SA["final_temperature"]:
            break
        candidate = proposal(state["current"], temperature, rng)
        try:
            loss, seconds = objective(candidate)
            if not np.isfinite(loss):
                loss = float("inf")
        except (FloatingPointError, ValueError) as exc:
            print("SA candidate failed:", exc)
            loss, seconds = float("inf"), 0.0
        delta = loss - state["current_loss"]
        accepted, probability = metropolis(delta, temperature, rng)
        if accepted:
            state["current"], state["current_loss"] = candidate, loss
        if loss < state["best_loss"]:
            state["best"], state["best_loss"] = candidate, loss
        state["evaluations"] += 1
        state["total_epochs"] += budget["sa_epochs"]
        state["total_seconds"] += seconds
        state["history"].append({"evaluation": state["evaluations"], "temperature": temperature,
                                 "validation_mse_scaled": loss, "best_mse_scaled": state["best_loss"],
                                 "accepted": accepted, "acceptance_probability": probability,
                                 "lstm_units_1": candidate["lstm_units_1"],
                                 "lstm_units_2": candidate["lstm_units_2"],
                                 "learning_rate": candidate["learning_rate"], "train_seconds": seconds})
        state["rng_state"] = rng.bit_generator.state
        atomic_json(checkpoint, state)
        pd.DataFrame(state["history"]).to_csv(history_path, index=False)
        print(f"SA {state['evaluations']}/{max_evals}: T={temperature:.3f}, val={loss:.6g}, best={state['best_loss']:.6g}")
    pd.DataFrame(state["history"]).to_csv(history_path, index=False)
    atomic_json(out / "best_sa_hyperparameters.json", {
        "hyperparameters": state["best"], "validation_mse_scaled": state["best_loss"],
        "evaluations": state["evaluations"], "total_search_epochs": state["total_epochs"],
        "total_search_seconds": state["total_seconds"], "mode": mode,
        "search_signature": signature})
    return state




## 9. Baseline training

Naive persistence uses only Copper from the previous session. Vanilla LSTM uses 32/32 units and learning rate 0.001 without tuning.

## 10. Paper SA-LSTM training

Published parameters: 39/111 units, learning rate 0.00005, batch 64, 100 epochs and dropout 0.2. All LSTMs use the same features, lookback, split, batch and epoch budget.

## 11. New SA-LSTM optimization and training

The full mode searches up to 100 evaluations with 20 epochs per trial, then trains selected parameters for 100 epochs. Cooling stops at temperature below 0.1. This differs from the paper's 650 reported scenarios and unclear 50-iteration schedule. Quick mode uses four evaluations and two epochs.

## 12. Evaluation

Report normalized MSE separately from squared price-unit MSE. SA is optimized on the primary 90/10 split and its parameters transfer unchanged to 92.5/7.5 and 95/5. Their test periods differ, so cross-ratio changes are descriptive.

In [14]:
def metrics(actual, predicted, scaler):
    actual = np.asarray(actual, dtype=float).ravel()
    predicted = np.asarray(predicted, dtype=float).ravel()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError("Invalid evaluation vectors")
    true_price = scaler.inverse_transform(actual[:, None]).ravel()
    pred_price = scaler.inverse_transform(predicted[:, None]).ravel()
    mse_price = mean_squared_error(true_price, pred_price)
    return {"mse_scaled": float(mean_squared_error(actual, predicted)),
            "mse_price2": float(mse_price), "rmse_price": float(math.sqrt(mse_price)),
            "mae_price": float(mean_absolute_error(true_price, pred_price)),
            "r2_price": float(r2_score(true_price, pred_price)) if len(actual) > 1 else float("nan")}


def save_curves(history, label, ratio, out):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history["loss"], label="train")
    ax.plot(history["val_loss"], label="validation")
    ax.set(xlabel="Epoch", ylabel="MSE, normalized", title=f"{label}, train ratio {ratio}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / f"loss_{label}_{ratio}.png", dpi=150)
    plt.close(fig)


def save_predictions(dates, actual, predicted, label, ratio, out):
    frame = pd.DataFrame({"Date": dates, "actual_price": actual, "predicted_price": predicted})
    frame.to_csv(out / f"predictions_{label}_{ratio}.csv", index=False)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(frame.Date, frame.actual_price, label="actual", linewidth=1)
    ax.plot(frame.Date, frame.predicted_price, label="predicted", linewidth=1)
    ax.set(xlabel="Target date", ylabel="Copper price", title=f"{label}, train ratio {ratio}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / f"prediction_{label}_{ratio}.png", dpi=150)
    plt.close(fig)


def plot_sa(state, out):
    hist = pd.DataFrame(state["history"])
    fig, axes = plt.subplots(2, 1, figsize=(8, 7))
    axes[0].plot(hist.evaluation, hist.validation_mse_scaled, label="candidate")
    axes[0].plot(hist.evaluation, hist.best_mse_scaled, label="best")
    axes[0].set(ylabel="Validation MSE, normalized")
    axes[0].legend()
    axes[1].plot(hist.evaluation, hist.temperature)
    axes[1].set(xlabel="Model evaluation", ylabel="Temperature")
    fig.tight_layout()
    fig.savefig(out / "sa_convergence.png", dpi=150)
    plt.close(fig)


def write_summary(results, state, data, mode, out):
    table = pd.DataFrame(results)
    primary = table[table.ratio == RATIOS[0]].set_index("model")
    lines = ["# Copper SA-LSTM research summary", "", f"Mode: **{mode}**. Date range: {data.index.min().date()} to {data.index.max().date()}; {len(data)} complete rows.", "",
             "Forecast target is next joint trading session. Inputs are the preceding 22 observations of WTI, Gold and Silver.",
             "Spearman and MinMax scalers were fit on training data only. Hyperparameters were selected using validation MSE only.",
             "", f"SA evaluated {state['evaluations']} candidates, trained {state['total_epochs']} search epochs, and spent {state['total_seconds']:.1f} seconds in search fits.",
             f"Selected hyperparameters: `{json.dumps(state['best'])}`.", "",
             "Metrics below come from this execution; price MSE is in squared copper price units.", "",
             "```csv", table.to_csv(index=False).strip(), "```", ""]
    if mode == "QUICK_MODE":
        lines += ["QUICK_MODE uses a short recent sample and abbreviated training. These numbers are diagnostic, not research conclusions.", ""]
    else:
        try:
            sa = float(primary.loc["new_sa_lstm", "mse_price2"])
            vanilla = float(primary.loc["vanilla_lstm", "mse_price2"])
            paper = float(primary.loc["paper_sa_lstm", "mse_price2"])
            lines += [f"On the primary 90/10 split, new SA vs vanilla MSE change: {(sa / vanilla - 1) * 100:+.2f}%.",
                      f"New SA vs paper preset MSE change: {(sa / paper - 1) * 100:+.2f}%.",
                      "These are descriptive comparisons from one seed and one test period; they do not prove a general SA advantage.", ""]
        except (KeyError, ZeroDivisionError):
            pass
    lines += ["Limits: original paper used Investing.com data from 1990–2009; this CSV is 2006–2026. The paper does not document forecast horizon or output activation, and Figure 12 implies four inputs despite naming three predictors. The three train ratios have different test periods. SA here uses a smaller, explicitly configured budget than 650 scenarios.", ""]
    (out / "research_summary.md").write_text("\n".join(lines), encoding="utf-8")




## 13. Visualization

Run All saves Spearman heatmap, learning curves, SA temperature and convergence, and actual versus predicted charts under `/kaggle/working/<mode>/`.

## 14. Comparative analysis

The final cell prints the full metric table and comparison pivot. Optional activation ablation uses validation only.

In [15]:
def run_experiment(data_path=DATA_PATH, mode=MODE, output_root=OUTPUT_ROOT):
    budget = run_config(mode)
    out = Path(output_root) / mode.lower()
    out.mkdir(parents=True, exist_ok=True)
    configure_tensorflow()
    data, quality = load_data(data_path, out, budget["max_rows"])
    print(f"{len(data)} complete rows: {data.index.min().date()} to {data.index.max().date()}")
    display(quality)
    splits = split_summary(data)
    splits.to_csv(out / "data_split_summary.csv", index=False)
    display(splits)
    primary = prepare(data, RATIOS[0])
    display(primary["spearman"]["Copper"].to_frame("Spearman_vs_Copper"))
    plot_spearman(primary["spearman"], out)
    # Invariant: search context does not even contain test labels or windows.
    assert "test" not in primary and "previous_copper" not in primary
    data_hash = hashlib.sha256(pd.util.hash_pandas_object(data, index=True).values.tobytes()).hexdigest()
    identity = {"data_sha256": data_hash, "ratio": RATIOS[0], "lookback": LOOKBACK,
                "features": FEATURES, "activation": DENSE_ACTIVATION}
    atomic_json(out / "experiment_manifest.json", {**identity, "mode": mode,
                "seed": SEED, "budget": budget, "tensorflow": __import__("tensorflow").__version__})

    def objective(hp):
        fit = train_model(primary, hp, budget["sa_epochs"])
        return fit["val_mse_scaled"], fit["train_seconds"]

    state = sa_search(objective, out, identity, mode, budget)
    plot_sa(state, out)
    display(pd.DataFrame([{"parameter": k, "value": v} for k, v in state["best"].items()]))
    results = []
    for ratio in RATIOS:
        ctx = prepare(data, ratio, include_test=True)
        actual_scaled = ctx["test"][1]
        dates = ctx["test"][2]
        scaler = ctx["y_scaler"]
        previous_scaled = scaler.transform(ctx["previous_copper"].reshape(-1, 1)).ravel()
        models = [("naive_persistence", None), ("vanilla_lstm", VANILLA_CONFIG),
                  ("paper_sa_lstm", PAPER_CONFIG), ("new_sa_lstm", state["best"])]
        for label, hp in models:
            if hp is None:
                predicted, seconds, val_mse = previous_scaled, 0.0, float("nan")
            else:
                fit = train_model(ctx, hp, budget["epochs"], predict_test=True)
                predicted, seconds, val_mse = fit["prediction_scaled"], fit["train_seconds"], fit["val_mse_scaled"]
                save_curves(fit["history"], label, ratio, out)
            scored = metrics(actual_scaled, predicted, scaler)
            actual_price = scaler.inverse_transform(actual_scaled.reshape(-1, 1)).ravel()
            predicted_price = scaler.inverse_transform(predicted.reshape(-1, 1)).ravel()
            save_predictions(dates, actual_price, predicted_price, label, ratio, out)
            results.append({"ratio": ratio, "model": label, "n_test": len(actual_scaled),
                            "val_mse_scaled": val_mse, **scored, "train_seconds": seconds,
                            "epochs": 0 if hp is None else budget["epochs"], "seed": SEED})
            pd.DataFrame(results).to_csv(out / "experiment_results.csv", index=False)
            print(f"{ratio:.3f} {label}: test RMSE={scored['rmse_price']:.4g}")
    table = pd.DataFrame(results)
    display(table)
    display(table.pivot(index="model", columns="ratio", values="mse_price2"))
    if RUN_ABLATION:
        ablation = []
        for activation in ("linear", "tanh", "sigmoid"):
            fit = train_model(primary, PAPER_CONFIG, budget["epochs"], activation=activation)
            ablation.append({"activation": activation, "val_mse_scaled": fit["val_mse_scaled"],
                             "train_seconds": fit["train_seconds"]})
        pd.DataFrame(ablation).to_csv(out / "activation_ablation_validation.csv", index=False)
        display(pd.DataFrame(ablation))
    write_summary(results, state, data, mode, out)
    print("Artifacts:", out)
    return table


In [16]:
results = run_experiment(DATA_PATH, MODE, OUTPUT_ROOT)

5231 complete rows: 2006-01-03 to 2026-10-29


,column,source_dtype,missing_or_nonfinite,duplicate_dates,invalid_dates,rows_source,first_date,last_date,rows_complete_used,rows_dropped_missing
0,Date,object,0,0,0,5231,2006-01-03,2026-10-29,5231,0
1,Copper,float64,0,0,0,5231,2006-01-03,2026-10-29,5231,0
2,Dow_Jones,float64,0,0,0,5231,2006-01-03,2026-10-29,5231,0
3,Nasdaq,float64,0,0,0,5231,2006-01-03,2026-10-29,5231,0
4,Dollar_Index,float64,0,0,0,5231,2006-01-03,2026-10-29,5231,0
5,TBill_10Y,float64,0,0,0,5231,2006-01-03,2026-10-29,5231,0
6,Gold,float64,0,0,0,5231,2006-01-03,2026-10-29,5231,0
7,Silver,float64,0,0,0,5231,2006-01-03,2026-10-29,5231,0
8,WTI,float64,0,0,0,5231,2006-01-03,2026-10-29,5231,0
9,Copper_Volume,int64,0,0,0,5231,2006-01-03,2026-10-29,5231,0


,ratio,segment,samples,first_target,last_target
0,0.900,train,4214,2006-02-03,2022-11-09
1,0.900,validation,471,2022-11-10,2024-09-26
2,0.900,test,524,2024-09-27,2026-10-29
3,0.925,train,4332,2006-02-03,2023-05-02
4,0.925,validation,484,2023-05-03,2025-04-07
5,0.925,test,393,2025-04-08,2026-10-29
6,0.950,train,4450,2006-02-03,2023-10-19
7,0.950,validation,497,2023-10-20,2025-10-14
8,0.950,test,262,2025-10-15,2026-10-29


,Spearman_vs_Copper
Copper,1.000000
Dow_Jones,0.038904
Nasdaq,0.043830
Dollar_Index,-0.437772
TBill_10Y,0.082866
Gold,0.412806
Silver,0.613513
WTI,0.711297
Copper_Volume,0.019671


SA 17/100: T=0.698, val=0.0467587, best=0.0171513
SA 18/100: T=0.663, val=0.031811, best=0.0171513
SA 19/100: T=0.663, val=0.0271336, best=0.0171513
SA 20/100: T=0.630, val=0.0362007, best=0.0171513
SA 21/100: T=0.630, val=0.017678, best=0.0171513
SA 22/100: T=0.599, val=0.0284324, best=0.0171513
SA 23/100: T=0.599, val=0.0326649, best=0.0171513
SA 24/100: T=0.569, val=0.0344102, best=0.0171513
SA 25/100: T=0.569, val=0.0234806, best=0.0171513
SA 26/100: T=0.540, val=0.0161815, best=0.0161815
SA 27/100: T=0.540, val=0.0211709, best=0.0161815
SA 28/100: T=0.513, val=0.0187399, best=0.0161815
SA 29/100: T=0.513, val=0.0187792, best=0.0161815
SA 30/100: T=0.488, val=0.0145026, best=0.0145026
SA 31/100: T=0.488, val=0.0341047, best=0.0145026
SA 32/100: T=0.463, val=0.0356673, best=0.0145026
SA 33/100: T=0.463, val=0.0305907, best=0.0145026
SA 34/100: T=0.440, val=0.0223818, best=0.0145026
SA 35/100: T=0.440, val=0.0325649, best=0.0145026
SA 36/100: T=0.418, val=0.0379585, best=0.0145026
SA

,parameter,value
0,lstm_units_1,9.000000
1,lstm_units_2,37.000000
2,learning_rate,0.000403


... (5 outputs left out)

0.950 naive_persistence: test RMSE=0.1014
0.950 vanilla_lstm: test RMSE=1.553
0.950 paper_sa_lstm: test RMSE=0.8404
0.950 new_sa_lstm: test RMSE=1.23


,ratio,model,n_test,val_mse_scaled,mse_scaled,mse_price2,rmse_price,mae_price,r2_price,train_seconds,epochs,seed
0,0.900,naive_persistence,524,NaN,0.000880,0.011926,0.109205,0.069313,0.982719,0.000000,0,42
1,0.900,vanilla_lstm,524,0.045729,0.049690,0.673466,0.820650,0.714608,0.024116,63.083489,100,42
2,0.900,paper_sa_lstm,524,0.007071,0.082417,1.117037,1.056900,0.961934,-0.618639,63.343777,100,42
3,0.900,new_sa_lstm,524,0.011590,0.022424,0.303918,0.551288,0.492054,0.559608,63.172422,100,42
4,0.925,naive_persistence,393,NaN,0.001023,0.013866,0.117754,0.074346,0.974396,0.000000,0,42
5,0.925,vanilla_lstm,393,0.018729,0.023012,0.311892,0.558473,0.493106,0.424077,63.785186,100,42
6,0.925,paper_sa_lstm,393,0.020267,0.095413,1.293168,1.137176,1.049971,-1.387891,65.395406,100,42
7,0.925,new_sa_lstm,393,0.021141,0.017878,0.242311,0.492251,0.435621,0.552562,64.412319,100,42
8,0.950,naive_persistence,262,NaN,0.000759,0.010289,0.101435,0.076662,0.962931,0.000000,0,42
9,0.950,vanilla_lstm,262,0.025140,0.177896,2.411101,1.552772,1.465174,-7.686790,64.467775,100,42


ratio,0.900,0.925,0.950
model,,,
naive_persistence,0.011926,0.013866,0.010289
new_sa_lstm,0.303918,0.242311,1.514035
paper_sa_lstm,1.117037,1.293168,0.706255
vanilla_lstm,0.673466,0.311892,2.411101


Artifacts: /kaggle/working/full_research_mode


## 15. Research conclusions

The generated `research_summary.md` reports only measured values and describes the limits of this reproduction.

In [17]:
from IPython.display import Markdown
display(Markdown((OUTPUT_ROOT / MODE.lower() / 'research_summary.md').read_text(encoding='utf-8')))

# Copper SA-LSTM research summary

Mode: **FULL_RESEARCH_MODE**. Date range: 2006-01-03 to 2026-10-29; 5231 complete rows.

Forecast target is next joint trading session. Inputs are the preceding 22 observations of WTI, Gold and Silver.
Spearman and MinMax scalers were fit on training data only. Hyperparameters were selected using validation MSE only.

SA evaluated 91 candidates, trained 1820 search epochs, and spent 1347.0 seconds in search fits.
Selected hyperparameters: `{"lstm_units_1": 9, "lstm_units_2": 37, "learning_rate": 0.00040267575055105107}`.

Metrics below come from this execution; price MSE is in squared copper price units.

```csv
ratio,model,n_test,val_mse_scaled,mse_scaled,mse_price2,rmse_price,mae_price,r2_price,train_seconds,epochs,seed
0.9,naive_persistence,524,,0.0008799044260694296,0.011925733552392816,0.10920500699323643,0.06931297618788193,0.9827190519303988,0.0,0,42
0.9,vanilla_lstm,524,0.04572851583361626,0.04968964882362761,0.6734657704015158,0.8206496026938146,0.7146083443093573,0.02411646597391237,63.08348907399977,100,42
0.9,paper_sa_lstm,524,0.0070707714185118675,0.0824172170229783,1.117036965862719,1.056899695270426,0.9619338958132232,-0.6186390308062419,63.34377658400035,100,42
0.9,new_sa_lstm,524,0.01158956903964281,0.022423690898422527,0.3039181926955122,0.5512877585213662,0.4920543599755888,0.5596082637344825,63.17242241199983,100,42
0.925,naive_persistence,393,,0.001023057813318401,0.013865954675134422,0.11775378836850398,0.07434606125015476,0.9743959204658009,0.0,0,42
0.925,vanilla_lstm,393,0.018729470670223236,0.023012040357317305,0.31189235292768264,0.5584732338507215,0.49310562975217626,0.42407740414805095,63.78518605699992,100,42
0.925,paper_sa_lstm,393,0.020267033949494362,0.09541253858344453,1.2931683021375948,1.137175581050523,1.0499714368280826,-1.3878906887249882,65.39540628500026,100,42
0.925,new_sa_lstm,393,0.021140508353710175,0.017878196518861165,0.24231109852882074,0.4922510523389673,0.4356210065710363,0.5525621723056051,64.41231943599996,100,42
0.95,naive_persistence,262,,0.0007591411131754777,0.01028897500247745,0.10143458484401388,0.07666222759252139,0.9629305537075125,0.0,0,42
0.95,vanilla_lstm,262,0.025139600038528442,0.17789581529860787,2.4111006042029857,1.5527719099091746,1.4651738233372593,-7.686789921422275,64.46777519900024,100,42
0.95,paper_sa_lstm,262,0.03019780106842518,0.05210890477660719,0.7062550155007314,0.8403897997362483,0.7053695522996206,-1.5445180262951754,65.10057355300023,100,42
0.95,new_sa_lstm,262,0.011575255542993546,0.11170849037336142,1.5140345385961582,1.2304611081201056,1.1397920969839344,-4.454811776678184,64.95839701900013,100,42
```

On the primary 90/10 split, new SA vs vanilla MSE change: -54.87%.
New SA vs paper preset MSE change: -72.79%.
These are descriptive comparisons from one seed and one test period; they do not prove a general SA advantage.

Limits: original paper used Investing.com data from 1990–2009; this CSV is 2006–2026. The paper does not document forecast horizon or output activation, and Figure 12 implies four inputs despite naming three predictors. The three train ratios have different test periods. SA here uses a smaller, explicitly configured budget than 650 scenarios.
